# 面试问题：搜索拼写纠错与 Query Rewrite 应该怎样设计？

可以直接复述的回答是：先判断是否需要改写，避免破坏本来正确的实体词。候选生成可以用编辑距离、键盘邻接、拼音或混淆集，候选排序再结合词频、上下文、品类和历史点击。Damerau-Levenshtein 比普通编辑距离多处理相邻字符转置。高频词不能无条件获胜，否则“苹果手表”的错字可能被改成更高频的“苹果手机”。生产系统应保留原查询、改写查询、置信度和触发原因，并允许结果页回退。下面用十个商品词典手写动态规划距离、候选打分和上下文门禁。

## 真实案例：电商搜索框的六条错字查询

词典包含商品词、搜索频次、品类和可返回商品。六条请求覆盖形近字、漏字以及一个高频歧义；数据为教学构造的脱敏日志，不能代表真实中文输入法分布。

In [1]:
import math  # 导入词频对数平滑函数
lexicon = [  # 定义十个带频次和品类的商品词条
    {"term": "蓝牙耳机", "frequency": 4000, "category": "audio", "product": "P-201"},  # 高频音频词
    {"term": "机械键盘", "frequency": 2500, "category": "office", "product": "P-202"},  # 办公外设词
    {"term": "咖啡机", "frequency": 1800, "category": "home", "product": "P-203"},  # 厨房电器词
    {"term": "扫地机器人", "frequency": 1600, "category": "home", "product": "P-204"},  # 清洁电器词
    {"term": "充电宝", "frequency": 3000, "category": "digital", "product": "P-205"},  # 便携电源词
    {"term": "运动相机", "frequency": 1200, "category": "camera", "product": "P-206"},  # 运动影像词
    {"term": "苹果手机", "frequency": 5000, "category": "mobile", "product": "P-207"},  # 高频手机词
    {"term": "苹果手表", "frequency": 800, "category": "wearable", "product": "P-208"},  # 低频穿戴词
    {"term": "无线鼠标", "frequency": 2800, "category": "office", "product": "P-209"},  # 办公鼠标词
    {"term": "空气净化器", "frequency": 1000, "category": "home", "product": "P-210"},  # 环境电器词
]  # 结束商品词典
query_logs = [  # 定义六条带上下文和人工期望的错字查询
    {"id": "S-01", "raw": "兰牙耳机", "context": "通勤", "expected": "蓝牙耳机"},  # 形近字替换
    {"id": "S-02", "raw": "机械健盘", "context": "办公", "expected": "机械键盘"},  # 键与健混淆
    {"id": "S-03", "raw": "咖非机", "context": "厨房", "expected": "咖啡机"},  # 啡与非混淆
    {"id": "S-04", "raw": "扫地机人", "context": "清洁", "expected": "扫地机器人"},  # 漏掉一个字符
    {"id": "S-05", "raw": "充点宝", "context": "旅行", "expected": "充电宝"},  # 电与点混淆
    {"id": "S-06", "raw": "苹果手坏", "context": "腕上", "expected": "苹果手表"},  # 与高频手机词同编辑距离的歧义
]  # 结束六条错字日志
term_by_text = {entry["term"]: entry for entry in lexicon}  # 建立词条文本索引
print("词典预览：term | frequency | category | product")  # 输出词典字段标题
for entry in lexicon:  # 逐条展示十个搜索词
    print(f"{entry['term']:6} | {entry['frequency']:5d} | {entry['category']:8} | {entry['product']}")  # 展示候选先验和商品映射
print("查询：", [(row["id"], row["raw"], row["context"], row["expected"]) for row in query_logs])  # 展示六条纠错目标

词典预览：term | frequency | category | product
蓝牙耳机   |  4000 | audio    | P-201
机械键盘   |  2500 | office   | P-202
咖啡机    |  1800 | home     | P-203
扫地机器人  |  1600 | home     | P-204
充电宝    |  3000 | digital  | P-205
运动相机   |  1200 | camera   | P-206
苹果手机   |  5000 | mobile   | P-207
苹果手表   |   800 | wearable | P-208
无线鼠标   |  2800 | office   | P-209
空气净化器  |  1000 | home     | P-210
查询： [('S-01', '兰牙耳机', '通勤', '蓝牙耳机'), ('S-02', '机械健盘', '办公', '机械键盘'), ('S-03', '咖非机', '厨房', '咖啡机'), ('S-04', '扫地机人', '清洁', '扫地机器人'), ('S-05', '充点宝', '旅行', '充电宝'), ('S-06', '苹果手坏', '腕上', '苹果手表')]


## Baseline / 基线：只做词典精确匹配

基线不改写，只有 query 本身完全等于词典词条时才返回商品。六条错字都没有结果，体现召回损失。

In [2]:
def exact_dictionary_lookup(query):  # 实现不纠错的词典精确查询
    entry = term_by_text.get(query)  # 直接按原查询查找词条
    return entry["product"] if entry else None  # 命中时返回商品否则返回空
baseline_rows = []  # 收集六条基线结果
print("query | raw | exact_product | expected_term")  # 输出精确匹配基线表头
for query in query_logs:  # 遍历六条错字请求
    product = exact_dictionary_lookup(query["raw"])  # 使用原查询执行词典查找
    baseline_rows.append((query["id"], product))  # 保存基线返回商品
    print(f"{query['id']} | {query['raw']:6} | {product} | {query['expected']}")  # 展示所有错字请求无结果
baseline_recall = sum(product is not None for query_id, product in baseline_rows) / len(baseline_rows)  # 计算原查询召回率
print(f"精确词典有结果比例={baseline_recall:.1%}")  # 汇总基线召回损失

query | raw | exact_product | expected_term
S-01 | 兰牙耳机   | None | 蓝牙耳机
S-02 | 机械健盘   | None | 机械键盘
S-03 | 咖非机    | None | 咖啡机
S-04 | 扫地机人   | None | 扫地机器人
S-05 | 充点宝    | None | 充电宝
S-06 | 苹果手坏   | None | 苹果手表
精确词典有结果比例=0.0%


## 核心实现：Damerau-Levenshtein 动态规划与候选生成

状态 `dp[i][j]` 表示源串前 i 个字符变到目标前 j 个字符的最小操作数。除插入、删除和替换外，相邻字符颠倒可以一步完成。

In [3]:
def damerau_levenshtein(source, target, return_matrix=False):  # 手写支持相邻转置的编辑距离
    rows = len(source) + 1  # 计算动态规划行数
    columns = len(target) + 1  # 计算动态规划列数
    matrix = [[0] * columns for _ in range(rows)]  # 初始化完整距离矩阵
    for row in range(rows):  # 初始化源串删除到空串的代价
        matrix[row][0] = row  # 每删除一个字符代价增加一
    for column in range(columns):  # 初始化空串插入到目标的代价
        matrix[0][column] = column  # 每插入一个字符代价增加一
    for row in range(1, rows):  # 逐个处理源串前缀
        for column in range(1, columns):  # 逐个处理目标串前缀
            substitution_cost = 0 if source[row - 1] == target[column - 1] else 1  # 判断当前字符是否需要替换
            deletion = matrix[row - 1][column] + 1  # 计算删除源字符的路径代价
            insertion = matrix[row][column - 1] + 1  # 计算插入目标字符的路径代价
            substitution = matrix[row - 1][column - 1] + substitution_cost  # 计算匹配或替换路径代价
            matrix[row][column] = min(deletion, insertion, substitution)  # 选择三种基础操作最小代价
            if row > 1 and column > 1 and source[row - 1] == target[column - 2] and source[row - 2] == target[column - 1]:  # 检查相邻字符颠倒
                matrix[row][column] = min(matrix[row][column], matrix[row - 2][column - 2] + 1)  # 用一步转置修正距离
    result = matrix[-1][-1]  # 读取完整源串到目标串距离
    return (result, matrix) if return_matrix else result  # 按教学需要返回距离或完整矩阵
def generate_candidates(raw_query, maximum_distance=2):  # 从十词词典生成编辑距离候选
    candidates = []  # 收集距离门限内词条
    for entry in lexicon:  # 遍历全部商品词条
        edit_distance = damerau_levenshtein(raw_query, entry["term"])  # 计算原查询到当前词的编辑距离
        if edit_distance <= maximum_distance:  # 只保留可解释的小编辑候选
            candidates.append((entry, edit_distance))  # 保存词条元数据和距离
    return candidates  # 返回未排序候选集合
example_distance, example_matrix = damerau_levenshtein("机械健盘", "机械键盘", True)  # 对形近字查询保留完整 DP 过程
print("机械健盘 -> 机械键盘 DP 矩阵：")  # 输出动态规划中间量标题
print("    ", "  ".join(["∅"] + list("机械键盘")))  # 展示矩阵目标字符列
for row_index, values in enumerate(example_matrix):  # 逐行展示 DP 状态
    row_label = "∅" if row_index == 0 else "机械健盘"[row_index - 1]  # 生成源字符行标签
    print(f"{row_label} | {values}")  # 展示每个前缀编辑代价
print("距离：", example_distance, "S-06 候选：", [(entry["term"], distance) for entry, distance in generate_candidates("苹果手坏")])  # 展示单错字和歧义候选

机械健盘 -> 机械键盘 DP 矩阵：
     ∅  机  械  键  盘
∅ | [0, 1, 2, 3, 4]
机 | [1, 0, 1, 2, 3]
械 | [2, 1, 0, 1, 2]
健 | [3, 2, 1, 1, 2]
盘 | [4, 3, 2, 2, 1]
距离： 1 S-06 候选： [('苹果手机', 1), ('苹果手表', 1)]


## 候选排序：词频先验加上下文品类证据

基础分数惩罚编辑距离并加入平滑词频；若请求上下文对应候选品类，再加 2 分。该规则是可审计教学实现，生产中可替换成学习排序模型。

In [4]:
context_category = {"通勤": "audio", "办公": "office", "厨房": "home", "清洁": "home", "旅行": "digital", "腕上": "wearable"}  # 定义请求上下文到品类的映射
def rank_candidates(raw_query, context, use_context):  # 对编辑候选执行可解释排序
    ranked = []  # 收集候选及各分项分数
    for entry, edit_distance in generate_candidates(raw_query):  # 遍历距离门限内候选
        frequency_score = 0.30 * math.log(entry["frequency"] + 1.0)  # 用对数频次形成温和先验
        context_score = 2.0 if use_context and context_category.get(context) == entry["category"] else 0.0  # 计算品类上下文奖励
        total_score = -2.0 * edit_distance + frequency_score + context_score  # 合并距离、词频和上下文
        ranked.append({"term": entry["term"], "product": entry["product"], "distance": edit_distance, "frequency": frequency_score, "context": context_score, "score": total_score})  # 保存完整候选解释
    ranked.sort(key=lambda item: (-item["score"], item["term"]))  # 按总分降序稳定排序
    return ranked  # 返回带分项的改写候选
example_ranked = rank_candidates("苹果手坏", "腕上", True)  # 对高频歧义请求执行上下文排序
print("S-06 candidate | edit | freq_score | context_score | total")  # 输出候选排序分项表头
for candidate in example_ranked:  # 遍历苹果歧义候选
    print(f"{candidate['term']} | {candidate['distance']} | {candidate['frequency']:.3f} | {candidate['context']:.1f} | {candidate['score']:.3f}")  # 展示低频手表为何胜出

S-06 candidate | edit | freq_score | context_score | total
苹果手表 | 1 | 2.006 | 2.0 | 2.006
苹果手机 | 1 | 2.555 | 0.0 | 0.555


## 失败案例与修正、六查询结果表

若只用编辑距离与词频，S-06 的两个候选距离都为 1，更高频的“苹果手机”会胜出。加入“腕上→wearable”上下文后才改写为“苹果手表”。

In [5]:
frequency_only = rank_candidates("苹果手坏", "腕上", False)  # 复现不使用上下文的高频偏差
context_fixed = rank_candidates("苹果手坏", "腕上", True)  # 使用品类上下文修正歧义
frequency_only_top = frequency_only[0]["term"]  # 提取错误的高频第一候选
context_fixed_top = context_fixed[0]["term"]  # 提取修正后的第一候选
result_rows = []  # 收集六条请求最终改写结果
print("id | raw | rewritten | product | expected | correct")  # 输出逐请求纠错结果表头
for query in query_logs:  # 遍历六条错字查询
    ranked = rank_candidates(query["raw"], query["context"], True)  # 生成并排序当前候选
    rewritten = ranked[0]["term"] if ranked else query["raw"]  # 无候选时安全保留原查询
    product = term_by_text[rewritten]["product"] if rewritten in term_by_text else None  # 用改写词真实查找商品
    correct = rewritten == query["expected"]  # 对照人工期望判断改写
    result_rows.append((query["id"], rewritten, product, correct))  # 保存逐查询结果
    print(f"{query['id']} | {query['raw']:6} | {rewritten:6} | {product} | {query['expected']:6} | {correct}")  # 展示查询改写和下游检索结果
rewrite_accuracy = sum(row[3] for row in result_rows) / len(result_rows)  # 计算六条教学查询改写准确率
print(f"S-06 仅词频={frequency_only_top}，加入上下文={context_fixed_top}")  # 展示失败与修正结论
print(f"精确词典有结果={baseline_recall:.1%}，改写准确率={rewrite_accuracy:.1%}")  # 对比同一六查询基线与主方案

id | raw | rewritten | product | expected | correct
S-01 | 兰牙耳机   | 蓝牙耳机   | P-201 | 蓝牙耳机   | True
S-02 | 机械健盘   | 机械键盘   | P-202 | 机械键盘   | True
S-03 | 咖非机    | 咖啡机    | P-203 | 咖啡机    | True
S-04 | 扫地机人   | 扫地机器人  | P-204 | 扫地机器人  | True
S-05 | 充点宝    | 充电宝    | P-205 | 充电宝    | True
S-06 | 苹果手坏   | 苹果手表   | P-208 | 苹果手表   | True
S-06 仅词频=苹果手机，加入上下文=苹果手表
精确词典有结果=0.0%，改写准确率=100.0%


## 结果解读

动态规划矩阵让每个候选的编辑代价可核验；候选排序则说明“距离最近”仍不等于“用户想要”。S-06 只有结合请求上下文才能抵抗高频词偏置。纠错结果随后真实进入商品词典查找，而不是停留在字符串断言。

## 生产边界

教学实现只含十个词和字符级距离。生产系统还要支持拼音、键盘邻接、品牌实体白名单、多词 query segmentation、个性化与地域词，并限制过度改写。需要离线评估 correction precision/recall、零结果率和点击收益，线上同时展示“仍搜索原词”的回退入口，记录模型与词典版本。

## 最小回归测试

In [6]:
assert len(lexicon) >= 6 and len(query_logs) >= 6  # 保证案例包含多个词条与查询
assert example_distance == 1  # 保证动态规划正确识别单字符替换
assert baseline_recall == 0.0  # 保证精确词典基线真实无法处理六条错字
assert frequency_only_top == "苹果手机"  # 保证高频先验歧义失败真实复现
assert context_fixed_top == "苹果手表"  # 保证品类上下文修正歧义
assert rewrite_accuracy == 1.0  # 保证六条教学查询均按人工期望改写
assert all(product is not None for query_id, rewritten, product, correct in result_rows)  # 保证改写后真实召回对应商品